# ACTUNEO AI: ask a question, get a worked solution

Type an actuarial question in plain words.

**A calculation** is worked out by ACTUNEO and set out the way a textbook does: what was given, each step with its formula, the substitution and the result, then the answer and the assumptions. Fixed rules read the numbers from your question; no language model does any arithmetic or writes any step.

**A question in words** ("What is the premium allocation approach?", "What does ACTUNEO do for pensions?") is answered by the language model, speaking as an actuary. It is given the matching passages of ACTUNEO's own documentation and uses them with its own knowledge. The passages are listed under the answer.

You can ask a question in several parts, and pass your own triangle, mortality table or cash flows.

**In a terminal** the same thing is:

```
actuneo-ai ask "What is the present value of 1000 due in 5 years at 10%?"
```

or `actuneo-ai chat` to ask one question after another.

## 1. What it can work out

In [12]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display
import actuneo.ai as ai

pd.set_option("display.max_colwidth", 120)
ai.topics()

,calculation,example
0,Present value of a single payment,What is the present value of 1000 due in 5 years at 10% interest?
1,Accumulated value of a single payment,"If I invest 5000 today at 9%, what will it grow to in 12 years?"
2,Present value of an annuity-certain,Present value of payments of 1000 at the end of each year for 10 years at 6%
3,Level monthly repayment on a loan,What is the monthly repayment on a loan of 50000 over 10 years at 8%?
4,Price of a fixed-interest bond,Price a 10-year bond with a 6% coupon at a yield of 7%.
5,Effective annual rate from a nominal rate,What is the effective annual rate of 12% convertible monthly?
6,Expectation of life,What is the life expectancy of a 45 year old woman?
7,Probability of survival,What is the probability that a 30 year old survives 20 years?
8,Value of a pension payable for life,Value a pension of 6000 a year from age 65 at 5% interest.
9,Net annual premium for a term assurance,Premium for a 20-year term assurance of 100000 for a 35 year old at 6% interest


## 2. Ask

The answer displays itself. Read *How the question was read* first: it shows which calculation was recognised and anything that was worked out from your wording.

In [13]:
ai.ask("What is the prsent valu of 14564 due in 5 years at 10% interest?")

> What is the prsent valu of 14564 due in 5 years at 10% interest?

*How the question was read:*

- Calculation: Present value of a single payment, chosen by the language model from your wording. Check that it is the one you meant.

### Present value of a single payment

**Given**

- amount S: `14,564.00`
- time n: `5 years`
- interest i: `10%`

**Working**

1. **Discount factor for one year**

   ```
   v = 1 / (1 + i)
     = 1 / 1.1
     = 0.909091
   ```
2. **Discount factor for 5 years**

   ```
   v^n
     = 0.909091^5
     = 0.620921
   ```
3. **Present value**

   ```
   PV = S x v^n
      = 14,564.00 x 0.620921
      = 9,043.10
   ```

**Answer:** The present value is 9,043.10.

**Check:** Accumulating 9,043.10 for 5 years at 10% gives 14,564.00, the amount due.

**Assumptions**

- Interest is compounded once a year at an effective rate.

In [14]:
ai.ask("What is the monthly repayment on a loan of 50,000 over 10 years at 8% reducing balances?")

> What is the monthly repayment on a loan of 50,000 over 10 years at 8% reducing balances?

*How the question was read:*

- Calculation: Level monthly repayment on a loan, recognised from your wording.

### Level monthly repayment on a loan

**Given**

- loan L: `50,000.00`
- interest i: `8% a year`
- term: `10 years, so n = 120 monthly payments`

**Working**

1. **Monthly rate of interest equivalent to the annual rate**

   ```
   j = (1 + i)^(1/12) - 1
     = 1.08^(1/12) - 1
     = 0.00643403
   ```
2. **Annuity factor for 120 monthly payments**

   ```
   a_n at rate j = (1 - (1 + j)^-n) / j
     = (1 - 0.463193) / 0.00643403
     = 83.4324
   ```
3. **Repayment, from: loan = repayment x annuity factor**

   ```
   X = L / a_n
     = 50,000.00 / 83.4324
     = 599.29
   ```
4. **First repayment split into interest and capital**

   ```
   interest = L x j = 50,000.00 x 0.00643403 = 321.70
   capital  = X - interest = 599.29 - 321.70 = 277.59
   ```

**Answer:** The repayment is 599.29 a month.

**Check:** Total repaid is 120 x 599.29 = 71,914.52, of which 21,914.52 is interest.

**Assumptions**

- 8% is taken as an effective annual rate, repayments are monthly in arrears, and the loan is repaid in full by the end.
- If 8% is instead a nominal rate convertible monthly, the monthly rate is i/12 = 0.00666667 and the repayment is 606.64.

### Your own question

Change the text and run the cell.

In [15]:
ai.ask("Value a pension of 6000 a year from age 65 at 5% interest.")

> Value a pension of 6000 a year from age 65 at 5% interest.

*How the question was read:*

- Calculation: Value of a pension payable for life, recognised from your wording.

### Value of a pension payable for life

**Given**

- age x: `65`
- sex: `male`
- pension: `6,000.00 a year, paid monthly in advance`
- interest i: `5%`

**Working**

1. **Discount factor**

   ```
   v = 1 / (1 + i)
     = 0.952381
   ```
2. **Annuity of 1 a year paid yearly in advance**

   ```
   a-due_x = sum over k of v^k x k_p_x   (k = 0, 1, 2, ...)
   k=0: v^0 x 0_p_x = 1.000000 x 1.000000 = 1.000000
   k=1: v^1 x 1_p_x = 0.952381 x 0.994943 = 0.947565
   k=2: v^2 x 2_p_x = 0.907029 x 0.989828 = 0.897803
   ...
   a-due_x = 13.9973
   ```
3. **Adjust for monthly payment**

   ```
   a-due_x(12) is approximately a-due_x - 11/24
     = 13.9973 - 0.4583 = 13.5390
   Exact, with deaths spread evenly over each year: 13.5336
   ```
4. **Value of the pension**

   ```
   value = pension x a-due_x(12)
         = 6,000.00 x 13.5336
         = 81,201.46
   ```

**Answer:** The pension is worth 81,201.46.

**Check:** The factor 13.53 means the pension is worth about 14 years' payments. Expectation of life at 65 on this table is 24.2 years; the factor is lower because of discounting.

**Assumptions**

- Mortality: Zimbabwe 2023. The published pensioner tables start at age 76, so the male assured lives rates are used up to age 75 and the post-retirement pensioner rates (males) after that. This join is ACTUNEO's, not a published table.
- Paid monthly in advance for life, with no increases, no guarantee period and no spouse's pension.
- Interest is an effective annual rate.

## 3. Questions to try

Each list is harder than the last. Run a cell to see every question answered in full, or copy one into the cell above.

### Stated plainly

In [16]:
PLAIN = [
    "What is the life expectancy of a 45 year old woman?",
    "What is the probability that a 30 year old survives 20 years?",
    "Premium for a 20-year term assurance of 100000 for a 35 year old at 6% interest",
    "Estimate the IBNR for the RAA triangle with the chain ladder method.",
    "What is the Mack standard error for the GenIns triangle?",
    "Price a 10-year bond with a 6% coupon at a yield of 7%.",
    "What is the effective annual rate of 12% convertible monthly?",
    "Present value of payments of 1000 at the end of each year for 10 years at 6%",
    "Convert 1,000,000 ZWL to ZiG.",
]
for question in PLAIN:
    display(ai.ask(question))

> What is the life expectancy of a 45 year old woman?

*How the question was read:*

- Calculation: Expectation of life, recognised from your wording.

### Expectation of life

**Given**

- age x: `45`
- sex: `female`

**Working**

1. **Survivors at age 45, from the table (out of 100,000 at age 20)**

   ```
   l_45 = 97,766.5
   ```
2. **Total years lived after that age by those survivors**

   ```
   T_x = L_x + L_(x+1) + ...   where L_y = l_y - d_y / 2
   T_45 = 4,221,417.8
   ```
3. **Complete expectation of life**

   ```
   e_x (complete) = T_x / l_x
     = 4,221,417.8 / 97,766.5
     = 43.18 years
   ```
4. **Curtate expectation (whole years only), for comparison**

   ```
   e_x = p_x + 2p_x + 3p_x + ...
       = 42.69 years
   The complete expectation is about half a year more.
   ```

**Answer:** A woman aged 45 is expected to live another 43.2 years, to about age 88.

**Check:** This is an average over the lives in the table, not a prediction for one person.

**Assumptions**

- Mortality: Zimbabwe 2023 male assured lives (no female assured lives table is published, so the male table is used).
- Deaths are spread evenly over each year of age.
- The table stops at age 100. Years lived beyond it are counted only if the table publishes them, so the figure may be slightly low.

> What is the probability that a 30 year old survives 20 years?

*How the question was read:*

- Calculation: Probability of survival, recognised from your wording.

### Probability of survival

**Given**

- age x: `30`
- period n: `20 years (to age 50)`

**Working**

1. **Survivors at each age, from the table (out of 100,000 at age 20)**

   ```
   l_30 = 99,253.5
   l_50 = 96,431.7
   ```
2. **Probability of surviving n years**

   ```
   n_p_x = l_(x+n) / l_x
     = 96,431.7 / 99,253.5
     = 0.971570
   ```
3. **Probability of dying within the period**

   ```
   n_q_x = 1 - n_p_x
     = 0.028430
   ```

**Answer:** The probability that a life aged 30 survives 20 years is 97.16%.

**Check:** Of every 1,000 lives aged 30, about 972 reach age 50.

**Assumptions**

- Mortality: Zimbabwe 2023 male assured lives.

> Premium for a 20-year term assurance of 100000 for a 35 year old at 6% interest

*How the question was read:*

- Calculation: Net annual premium for a term assurance, recognised from your wording.

### Net annual premium for a term assurance

**Given**

- age x: `35`
- term n: `20 years`
- sum assured S: `100,000.00`
- interest i: `6%`

**Working**

1. **Discount factor**

   ```
   v = 1 / (1 + i)
     = 0.943396
   ```
2. **Expected present value of 1 paid at the end of the year of death, if within the term**

   ```
   A = sum over k of v^(k+1) x k_p_x x q_(x+k)   (k = 0 to n-1)
   k=0: v^1 x 0_p_x x q_35 = 0.943396 x 1.000000 x 0.000662 = 0.00062415
   k=1: v^2 x 1_p_x x q_36 = 0.889996 x 0.999338 x 0.000766 = 0.00068093
   ...
   A = 0.021572
   ```
3. **Expected present value of 1 a year paid in advance while alive, for the term**

   ```
   a-due = sum over k of v^k x k_p_x   (k = 0 to n-1)
   a-due = 12.0224
   ```
4. **Equation of value: premiums = benefits**

   ```
   P x a-due = S x A
   P = S x A / a-due
     = 100,000.00 x 0.021572 / 12.0224
     = 179.43
   ```

**Answer:** The net premium is 179.43 a year.

**Check:** The premium is 0.179% of the sum assured each year.

**Assumptions**

- Mortality: Zimbabwe 2023 male assured lives.
- The sum assured is paid at the end of the year of death. Premiums are paid yearly in advance for the term, or until earlier death.
- This is a net premium: no expenses, profit or tax.

> Estimate the IBNR for the RAA triangle with the chain ladder method.

*How the question was read:*

- Calculation: Chain ladder reserve, recognised from your wording.

### Chain ladder reserve

**Given**

- triangle: `RAA (reinsurance general liability, a standard example)`
- origin years: `10`

**Working**

1. **Development factors, one for each step from one development year to the next**

   ```
   f_j = (sum of claims at development j+1) / (sum of claims at development j),
         over the origin years observed at both
   1-2: 2.9994
   2-3: 1.6235
   3-4: 1.2709
   4-5: 1.1717
   5-6: 1.1134
   6-7: 1.0419
   7-8: 1.0333
   8-9: 1.0169
   9-10: 1.0092
   ```
2. **Project each origin year to its ultimate**

   ```
   ultimate = latest claims x the factors still to come
   For 1990: 2,063 / 0.112105 = 18,402
   (dividing by the proportion developed is the same as multiplying by the factors)
   ```
3. **Reserve for each origin year**

   ```
   IBNR = ultimate - latest
     origin         latest       ultimate           IBNR
       1981         18,834         18,834              0
       1982         16,704         16,858            154
       1983         23,466         24,083            617
       1984         27,067         28,703          1,636
       1985         26,180         28,927          2,747
       1986         15,852         19,501          3,649
       1987         12,314         17,749          5,435
       1988         13,112         24,019         10,907
       1989          5,395         16,045         10,650
       1990          2,063         18,402         16,339
   ```
4. **Total**

   ```
   IBNR = 52,135
   ```

**Answer:** The chain ladder reserve is 52,135.

**Check:** The most recent origin year carries 31% of the reserve, and is the least certain.

**Assumptions**

- Development factors are volume-weighted averages over all origin years.
- No tail factor: claims are taken as fully developed at the last development year.
- The method assumes later years will develop as earlier years did. Check that before relying on the figure: see actuneo.loss_reserving.diagnose.

> What is the Mack standard error for the GenIns triangle?

*How the question was read:*

- Calculation: Mack standard error of the chain ladder reserve, recognised from your wording.

### Mack standard error of the chain ladder reserve

**Given**

- triangle: `GenIns (the Taylor and Ashe data, a standard example)`

**Working**

1. **The reserve, from the chain ladder**

   ```
   IBNR = 18,680,856
   ```
2. **Standard error for each origin year, from Mack's formula**

   ```
   It combines the variance of each development factor with the size of the
   claims still to develop.
     origin           IBNR      std error       cv
          1              0              0        -
          2         94,634         75,535      80%
          3        469,511        121,699      26%
          4        709,638        133,549      19%
          5        984,889        261,406      27%
          6      1,419,459        411,010      29%
          7      2,177,641        558,317      26%
          8      3,920,301        875,328      22%
          9      4,278,972        971,258      23%
         10      4,625,811      1,363,155      29%
   ```
3. **Total standard error, split into its two sources**

   ```
   process risk (claims are random)          = 1,878,292
   parameter risk (factors are estimates)    = 1,568,532
   standard error = sqrt(process^2 + parameter^2)
                  = sqrt(1,878,292^2 + 1,568,532^2)
                  = 2,447,095
   ```
4. **As a proportion of the reserve**

   ```
   cv = standard error / IBNR
      = 2,447,095 / 18,680,856
      = 13.1%
   ```

**Answer:** The standard error is 2,447,095, which is 13% of the reserve of 18,680,856.

**Check:** Roughly, a reserve of 18,680,856 plus or minus 2,447,095.

**Assumptions**

- Mack's 1993 distribution-free model, with his estimate of the last variance parameter.
- The total is not the sum of the rows: origin years share the same estimated factors, so their errors are correlated.
- It measures uncertainty if the chain ladder assumptions hold. It does not allow for the method itself being wrong.

> Price a 10-year bond with a 6% coupon at a yield of 7%.

*How the question was read:*

- Calculation: Price of a fixed-interest bond, recognised from your wording.

### Price of a fixed-interest bond

**Given**

- coupon: `6% a year, paid half-yearly`
- term n: `10 years`
- yield i: `7% a year`
- redemption: `100 per 100 nominal (at par)`

**Working**

1. **Each half-yearly coupon per 100 nominal**

   ```
   coupon = 100 x coupon rate / 2
          = 100 x 0.06 / 2
          = 3.0000
   ```
2. **Half-yearly rate equivalent to the yield**

   ```
   j = (1 + i)^(1/2) - 1
     = 1.07^0.5 - 1
     = 0.03440804
   ```
3. **Present value of the 20 coupons**

   ```
   a_2n at rate j = (1 - (1 + j)^-2n) / j
     = 14.2888
   coupons = 3.0000 x 14.2888 = 42.8665
   ```
4. **Present value of the redemption payment**

   ```
   100 x v^n = 100 x (1 + i)^-n
     = 100 x 0.508349
     = 50.8349
   ```
5. **Price**

   ```
   P = coupons + redemption
     = 42.8665 + 50.8349
     = 93.7014
   ```

**Answer:** The price is 93.70 per 100 nominal.

**Check:** The price is below par because the coupon rate is below the yield.

**Assumptions**

- Coupons are paid half-yearly in arrears, the next one in six months, and the bond is redeemed at par.
- 7% is an effective annual yield. No tax.
- If the yield is instead a nominal rate convertible half-yearly, the half-yearly rate is i/2 = 0.035000 and the price is 92.89.

> What is the effective annual rate of 12% convertible monthly?

*How the question was read:*

- Calculation: Effective annual rate from a nominal rate, recognised from your wording.

### Effective annual rate from a nominal rate

**Given**

- nominal rate i(p): `12% a year`
- convertible: `monthly (p = 12)`

**Working**

1. **Rate for each period**

   ```
   i(p) / p
     = 0.12 / 12
     = 0.01000000
   ```
2. **Compound over a year**

   ```
   i = (1 + i(p)/p)^p - 1
     = 1.01000000^12 - 1
     = 0.12682503
   ```

**Answer:** The effective annual rate is 12.6825%.

**Check:** The effective rate exceeds the nominal rate by 0.6825% because interest is earned on interest within the year.

**Assumptions**

- Interest is added at the end of each period and earns interest thereafter.

> Present value of payments of 1000 at the end of each year for 10 years at 6%

*How the question was read:*

- Calculation: Present value of an annuity-certain, recognised from your wording.

### Present value of an annuity-certain

**Given**

- payment P: `1,000.00 a year, at the end of each year`
- term n: `10 years`
- interest i: `6%`

**Working**

1. **Discount factors**

   ```
   v = 1 / (1 + i)
     = 0.943396
   v^n = 0.943396^10 = 0.558395
   ```
2. **Annuity factor**

   ```
   a_n = (1 - v^n) / i
       = (1 - 0.558395) / 0.06
       = 7.360087
   ```
3. **Present value**

   ```
   PV = P x a_n
      = 1,000.00 x 7.360087
      = 7,360.09
   ```

**Answer:** The present value is 7,360.09.

**Check:** The 10 payments total 10,000.00 before discounting.

**Assumptions**

- Payments are certain (not dependent on survival) and made at the end of each year.
- Interest is an effective annual rate.

> Convert 1,000,000 ZWL to ZiG.

*How the question was read:*

- Calculation: Zimbabwe dollars to Zimbabwe Gold, recognised from your wording.

### Zimbabwe dollars to Zimbabwe Gold

**Given**

- amount: `ZWL 1,000,000.00`

**Working**

1. **Conversion rate set in April 2024**

   ```
   1 ZiG = 2,498.7242 ZWL
   ```
2. **Convert**

   ```
   ZiG = ZWL / rate
       = 1,000,000.00 / 2,498.7242
       = 400.20
   ```

**Answer:** ZWL 1,000,000.00 is ZiG 400.20.

**Assumptions**

- The rate at which Zimbabwe dollar balances were converted when ZiG replaced the Zimbabwe dollar (Statutory Instrument 60 of 2024). It is not a market exchange rate.

### Said the way people say it

Different words from the textbook's, amounts written as `120k` or `5 million`, a rate given in words.

In [17]:
NATURAL = [
    "A bank lends me £1,000 for three years, repaid by three equal payments, one at the end of each year. The bank charges an effective rate of 7% per annum. What is each payment?",
]
for question in NATURAL:
    display(ai.ask(question))

> A bank lends me £1,000 for three years, repaid by three equal payments, one at the end of each year. The bank charges an effective rate of 7% per annum. What is each payment?

This looks like: Level monthly repayment on a loan.
To work it out I also need: the term of the loan in years.

For example: "What is the monthly repayment on a loan of 50000 over 10 years at 8%?"

### Advanced: something has to be worked out first

- "alive at 70" from age 45 is a period of 25 years
- 750 a month is 9,000 a year
- 7.5% from the member and 10% from the employer is 17.5% in total
- a 20-year bond issued 5 years ago has 15 years left
- a detail that is not part of the calculation (the borrower's age) must be ignored

*How the question was read* shows each of these so you can check it.

In [ ]:
ADVANCED = [
    "If I am 45 now, what are the odds I am still alive at 70?",
    "A widow aged 63 will receive 750 a month for life. What is that worth at 6%?",
    "I am 28, earn 18,000 a year and retire at 60. I put in 7.5% and my employer adds 10%. Returns are 8%. How big is my pot?",
    "A bond issued 5 years ago with a 20-year term pays 8%. Yields are 6%. What is the price now?",
    "Our Harare branch lent 80,000 to a 40 year old teacher over 20 years at 12%. Monthly instalment?",
    "I am 30, retire at 60, salary of 12000 with salary increases of 4%, contributing 10%, return 7%. Retirement fund?",
    "Present value of payments of 2,500 at the start of each year for 20 years at 7.5%",
    "What is the effective annual rate of 18% nominal compounded quarterly?",
]
for question in ADVANCED:
    display(ai.ask(question))

### Mortality basis

Mortality questions use the **Zimbabwe 2023 tables** unless you name another basis. Say "South African" for the ASSA tables or "AM92" for the UK examination table. The assumptions under each answer say exactly which table was used, including where the published Zimbabwean tables have gaps.

In [ ]:
display(ai.ask("What is the life expectancy of a 40 year old man?"))
display(ai.ask("What is the life expectancy of a 40 year old man on AM92?"))
display(ai.ask("Value a pension of 6000 a year from age 65 at 5% interest."))
display(ai.ask("Value a pension of 6000 a year from age 65 at 5% interest, South African tables"))

## 4. A question in several parts

Parts labelled (a), (b), (c), separate sentences that each ask something, or "... and what is ..." are split and answered one by one. What the opening sentence gives (here the age) is carried into every part.

In [ ]:
ai.ask("A man is aged 40. (a) What is his life expectancy? (b) What is the probability he survives 20 years? (c) What is the premium for a 10-year term assurance of 50000 at 6%?")

In [ ]:
ai.ask("What is the chain ladder reserve for RAA and GenIns?")

## 5. Your own data

Pass it with `data=`. A table of claims is read as a run-off triangle (origin years in rows, development years in columns, cumulative, blank where not yet observed). A table with columns `age` and `qx` is a mortality table. A plain list is yearly cash flows, the first one now. A CSV or Excel file path works too, and in a terminal: `actuneo-ai ask "IBNR reserve?" --data my_triangle.csv`.

In [ ]:
my_triangle = pd.DataFrame(
    [[1200, 1900, 2250, 2400],
     [1350, 2100, 2500, None],
     [1500, 2400, None, None],
     [1650, None, None, None]],
    index=[2022, 2023, 2024, 2025], columns=[1, 2, 3, 4])

display(ai.ask("What is the chain ladder reserve for my triangle?", data=my_triangle))
display(ai.ask("How uncertain is that reserve?", data=my_triangle))

In [ ]:
import numpy as np

ages = np.arange(20, 111)
my_table = pd.DataFrame({"age": ages, "qx": np.minimum(0.0004 * 1.095 ** (ages - 20), 1.0)})
my_table.loc[my_table["age"] == 110, "qx"] = 1.0

display(ai.ask("What is the life expectancy at age 60?", data=my_table))
display(ai.ask("Value a pension of 12,000 a year from age 60 at 6%", data=my_table))

In [ ]:
project = [-10_000, 2_500, 3_000, 3_500, 4_000]

display(ai.ask("What is the NPV of this project at 9%?", data=project))
display(ai.ask("What is the NPV at 8% of cash flows of -1000, 300, 400, 500?"))

### IFRS 17, premium allocation approach

In [ ]:
display(ai.ask("IFRS 17 PAA: premium of 1200 for a 12-month policy with acquisition costs of 180. What is the LRC after 3 months?"))

## 6. Questions in words

These go to the language model, which answers as an actuary from its own knowledge and from the passages of ACTUNEO's documentation listed under each answer. Install a model first, in a terminal: `actuneo-ai setup`.

**How good these answers are depends on the model.** The smallest model (0.4 GB) is accurate about ACTUNEO when the passages give it the facts, and often wrong about general actuarial ideas. A 4B model or larger is far better. Every such answer is marked as written by the model. Without a model, the passages themselves are shown.

In [ ]:
installed = ai.status(quiet=True)["local_model"]
print("Model installed:", installed["name"] if installed else "none")

for question in [
    "What is ACTUNEO?",
    "What does ACTUNEO do for IFRS 17?",
    "What are the limitations of the pensions module?",
    "How do I load a Zimbabwe mortality table?",
    "What is the premium allocation approach?",
    "What is a pension?",
    "What is IPEC?",
    "Explain the chain ladder method and when it fails.",
]:
    display(ai.ask(question))

## 7. When it cannot calculate

It says what is missing instead of guessing, and refuses values that make no sense.

In [ ]:
for question in [
    "What is the monthly repayment on my loan?",                  # details missing
    "Value my pension.",                                          # details missing
    "What is the life expectancy of a 10-year bond?",             # a trap
    "What is the life expectancy of a 400 year old man?",         # not sensible
    "What is the probability that a 5 year old survives 10 years?",   # off the table
    "Good morning",
]:
    display(ai.ask(question))

## 8. Using the answer in your own work

The answer is an object. `value` is the number, `arguments` is what the calculation was given, and `explain()` is the working as plain text for a report or an email.

In [ ]:
answer = ai.ask("Premium for a 20-year term assurance of 100000 for a 35 year old at 6% interest")
print("calculation:", answer.solver)
print("arguments  :", answer.arguments)
print("value      :", round(answer.value, 2))
print()
print(answer.explain())

In [ ]:
# Sensitivity: the same question at several rates of interest
rows = []
for rate in (4, 5, 6, 7, 8):
    result = ai.ask(f"Value a pension of 6000 a year from age 65 at {rate}% interest")
    rows.append({"interest": f"{rate}%", "value of pension": round(result.value)})
pd.DataFrame(rows)

## 9. Check it yourself

Each test is a question with the calculation and values a correct reading gives. The mark is **ok** when both are right. This runs with the rules only, so the score does not depend on which model you have installed.

In [ ]:
from actuneo.ai.benchmark import arguments_match

TESTS = [
    ("What is the life expectancy of a 45 year old woman?", "life_expectancy", {"age": 45, "sex": "F"}),
    ("What is the probability that a 30 year old survives 20 years?", "survival_probability", {"age": 30, "years": 20}),
    ("Value a pension of 6000 a year from age 65 at 5% interest.", "life_annuity", {"age": 65, "annual_amount": 6000, "interest_rate": 0.05}),
    ("Premium for a 20-year term assurance of 100000 for a 35 year old at 6% interest", "term_assurance_premium", {"age": 35, "term": 20, "sum_assured": 100000, "interest_rate": 0.06}),
    ("What's the IBNR on GenIns?", "chain_ladder_reserve", {"triangle": "GenIns"}),
    ("How uncertain is the chain ladder reserve for RAA?", "mack_standard_error", {"triangle": "RAA"}),
    ("Borrowing 120k over 15 years at eight and a half percent. What do I pay each month?", "loan_repayment", {"amount": 120000, "interest_rate": 0.085, "years": 15}),
    ("A 15-year bond pays a 9% coupon and yields are 11%. What should it trade at?", "bond_price", {"coupon_rate": 0.09, "years": 15, "yield_rate": 0.11}),
    ("What is 25,000 payable in 12 years worth today at 7.5%?", "present_value", {"amount": 25000, "years": 12, "interest_rate": 0.075}),
    ("How much is ZWL 5 million in ZiG?", "zwl_to_zig", {"amount": 5000000}),
    ("If I am 45 now, what are the odds I am still alive at 70?", "survival_probability", {"age": 45, "years": 25}),
    ("A widow aged 63 will receive 750 a month for life. What is that worth at 6%?", "life_annuity", {"age": 63, "annual_amount": 9000, "interest_rate": 0.06, "sex": "F"}),
    ("I am 28, earn 18,000 a year and retire at 60. I put in 7.5% and my employer adds 10%. Returns are 8%. How big is my pot?", "retirement_fund", {"age": 28, "retirement_age": 60, "salary": 18000, "contribution_rate": 0.175, "investment_return": 0.08}),
    ("A bond issued 5 years ago with a 20-year term pays 8%. Yields are 6%. What is the price now?", "bond_price", {"coupon_rate": 0.08, "years": 15, "yield_rate": 0.06}),
    ("Our Harare branch lent 80,000 to a 40 year old teacher over 20 years at 12%. Monthly instalment?", "loan_repayment", {"amount": 80000, "interest_rate": 0.12, "years": 20}),
    # these should NOT be calculated
    ("What is the monthly repayment on my loan?", None, {}),
    ("What is the life expectancy of a 10-year bond?", None, {}),
    ("Good morning, can you help me?", None, {}),
]

rows = []
for question, expected, expected_arguments in TESTS:
    answer = ai.ask(question, use_model=False)
    chosen = answer.solver if answer.solved else None
    right = chosen == expected and (expected is None or arguments_match(expected_arguments, answer.arguments))
    rows.append({"ok": "ok" if right else "NO", "question": question, "expected": expected,
                 "got": chosen, "result": answer.solution.answer if answer.solved else f"({answer.kind})"})
results = pd.DataFrame(rows)
print(f"{(results['ok'] == 'ok').sum()} of {len(results)} right")
results

## 10. Unusual wording

When the wording does not contain the phrases a calculation is recognised by, the rules cannot choose. With a model installed it is asked to choose, and the answer says so: *chosen by the language model. Check that it is the one you meant.* A small model often cannot tell either.

In [ ]:
UNUSUAL = [
    "What would a 10 year gilt paying 6% cost me if rates are 7%?",
    "I have 5000 and leave it in the bank for 12 years at 9%. What do I end up with?",
    "How many more birthdays will a 60 year old man see on average?",
]
installed = ai.status(quiet=True)["local_model"]
print("Model installed:", installed["name"] if installed else "none")
for question in UNUSUAL:
    rules = ai.ask(question, use_model=False)
    helped = ai.ask(question)
    print(f"\n{question}")
    print(f"   rules only : {rules.solver if rules.solved else rules.kind}")
    print(f"   with model : {helped.solver if helped.solved else helped.kind}")

## 11. Add your own calculation

A calculation is a function that returns a `Solution`, registered with the phrases that identify it and the quantities it needs. Here is the present value of a perpetuity, `P / i`.

In [ ]:
from actuneo.ai.solutions import Solution, Step, money, percent
from actuneo.ai.solvers import Param, Solver, register


def perpetuity(payment, interest_rate):
    value = payment / interest_rate
    return Solution(
        "Present value of a perpetuity",
        {"payment P": f"{money(payment)} a year, for ever", "interest i": percent(interest_rate)},
        [Step("A perpetuity in arrears pays P at the end of every year for ever",
              ["PV = P x (v + v^2 + v^3 + ...)", "   = P / i"]),
         Step("Substitute", [f"PV = {money(payment)} / {interest_rate:.6g}", f"   = {money(value)}"])],
        f"The present value is {money(value)}.", value,
        ["Payments are made at the end of each year and never stop."],
        f"Invested at {percent(interest_rate)}, {money(value)} earns {money(payment)} a year without touching the capital.")


register(Solver(
    "perpetuity", "Present value of a perpetuity",
    "Present value of a level payment made every year for ever (a perpetuity)",
    ((r"perpetuit", 5), (r"for ever|forever|in perpetuity", 4)),
    (Param("payment", "amount", "the yearly payment", low=0),
     Param("interest_rate", "rate", "the interest rate", low=0.0001, high=1)),
    perpetuity, "What is the present value of a perpetuity of 500 a year at 4%?"))

ai.ask("What is 500 a year for ever worth at 4%?")